# Transformacion Map()

In [4]:
!pip install findspark
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Transformaciones_1").getOrCreate()

In [6]:
spark

In [5]:
sc = spark.sparkContext

In [7]:
rdd = sc.parallelize([1,2,3,4,5])

In [9]:
#para crear funciones map debemos emplear funciones lambda

rdd_resta = rdd.map(lambda x: x - 1)

rdd_resta.collect()

[0, 1, 2, 3, 4]

Vemos que se le resto al RDD principal con la funcion map obtenemos una lista restandole 1 a cada elemento

In [10]:
rdd_par = rdd.map(lambda x: x % 2 ==  0)

rdd_par.collect()

[False, True, False, True, False]

Nos devolvio TRUE o FALSE en dependencia si el numero que existe en el RDD original es divisible o no por 2

In [11]:
rdd_texto = sc.parallelize(["Python", "es", "un", "lenguaje", "de", "programacion"])

rdd_texto.collect()

['Python', 'es', 'un', 'lenguaje', 'de', 'programacion']

In [12]:
rdd_mayus = rdd_texto.map(lambda x: x.upper())

rdd_mayus.collect()

['PYTHON', 'ES', 'UN', 'LENGUAJE', 'DE', 'PROGRAMACION']

In [13]:
rdd_hola = rdd_texto.map(lambda x: "Hola " + x)

rdd_hola.collect()

['Hola Python',
 'Hola es',
 'Hola un',
 'Hola lenguaje',
 'Hola de',
 'Hola programacion']

In [ ]:
# siempre nos devuelve con esta transformacion un nuevo RDD

# Transformacion flatMap()

In [14]:
rdd_1 = sc.parallelize([1,2,3,4,5])

rdd_1.collect()

[1, 2, 3, 4, 5]

In [16]:
rdd_cuadrado = rdd_1.map(lambda x: (x, x**2))

rdd_cuadrado.collect()

[(1, 1), (2, 4), (3, 9), (4, 16), (5, 25)]

Podemos observar que nos devuelve una lista con su numero correspondiente y su cuadrado en la segunda posición y asi sucesivamente

In [25]:
# Flatten para que nos lo devuelva como elementos independientes no como una lista con tuplas (hace un aplanado flatten del resultado)

rdd_cuadrado_flat = rdd_1.flatMap(lambda x: (x, x**2))

rdd_cuadrado_flat.collect()

[1, 1, 2, 4, 3, 9, 4, 16, 5, 25]

In [29]:
rdd_texto = sc.parallelize(["Juan", "Francisco","Maria","Lucia"])

rdd_mayuscula = rdd_texto.flatMap(lambda x: (x, x.upper()))

rdd_mayuscula.collect()

['Juan', 'JUAN', 'Francisco', 'FRANCISCO', 'Maria', 'MARIA', 'Lucia', 'LUCIA']

# Transformacion filter()

In [30]:
rdd_2 = sc.parallelize([1,2,3,4,5,6,7,8,9,10])

rdd_2.collect()

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

In [31]:
rdd_par = rdd_2.filter(lambda x: x % 2 == 0)

rdd_par.collect()

[2, 4, 6, 8, 10]

El resto de la divicion por 2 de cada uno de los elementos de lso RDD sean iguales a 0 y nos da como resultado numeros que sean pares

In [32]:
rdd_impar = rdd_2.filter(lambda x: x % 2 != 0)

rdd_impar.collect()

[1, 3, 5, 7, 9]

In [44]:
rdd_texto_2 = sc.parallelize(["Python", "es", "un", "lenguaje","lengua", "de", "programacion"])

rdd_texto_2.collect()

['Python', 'es', 'un', 'lenguaje', 'lengua', 'de', 'programacion']

In [34]:
rdd_con_o = rdd_texto_2.filter(lambda x: "o" in x)

rdd_con_o.collect()

['Python', 'programacion']

In [39]:
rdd_con_u = rdd_texto_2.filter(lambda x: x.startswith("p"))

rdd_con_u.collect()

['programacion']

Transformacion filter con un operador

In [45]:
rdd_filro = rdd_texto_2.filter(lambda x: x.startswith("l") and x.find("e") == 1)

rdd_filro.collect()

['lenguaje', 'lengua']

# Transformacion coalesce()

In [47]:
rdd_3 = sc.parallelize([1,2,3,4,5,6,7,8,9,10], 10)

rdd_3.getNumPartitions()

10

In [50]:
#IMPORTANTE siempre crear una variable porqque los RDDs son inmutables ( coalesce es una transgormacion asi que por ende se crea un RDD nuevo)

rdd_2_coalesce = rdd_3.coalesce(2)

rdd_3_coalesce.getNumPartitions()

2

# Transformacion repartition()

In [51]:
rdd_4 = sc.parallelize([1,2,3,4,5,6,7,], 5)

rdd_4.getNumPartitions()

5

In [53]:
# La vamos a emplear para disminuir o aumentar el numero de particiones de un RDD

rdd_partition = rdd_4.repartition(10)

rdd_partition.getNumPartitions()

10

## Cual es la diferencia entre Coalesce y Repartition?

**coalesce** se usa solo para refucir el numero de particiones. Esta es una version optimizada de **repartition** donde el movimiento de los datos a traves de las particiones es menor

**Importante**: repartition y coalesce son operaciones muy costosas ya que mezclan los datos en muchas particiones, por lo tanto, intente, minimizar el uso de estos como sea posible  

##  Transformacion reduceByKey()

In [54]:
rdd_5 = sc.parallelize([
("casa", 2),
("paerque", 1),
("que", 5),
("casa", 1),
("escuela", 2),
("casa", 1),
("que", 1)
])

rdd_5.collect()

[('casa', 2),
 ('paerque', 1),
 ('que', 5),
 ('casa', 1),
 ('escuela', 2),
 ('casa', 1),
 ('que', 1)]

In [56]:
# podemos usar suma resta multiplicacion etc todo depende de los elementos que estemos trabajando en este caso hizo un summary por sus llaves (Los elementos repetidos los elimino para poder unirlos)

rdd_reduced = rdd_5.reduceByKey(lambda x,y: x + y)

rdd_reduced.collect()

[('casa', 4), ('paerque', 1), ('que', 6), ('escuela', 2)]